# Bandcamp Artist Discog Scraper
Based on:
- dbeley's [bandcamp-library-scraper.py](https://github.com/dbeley/bandcamp-library-scraper/blob/main/bandcamp-library-scraper.py)
- nahnhh's [webcrawl2_movie details.ipynb](https://github.com/nahnhh/top-movies-2005-2025/blob/main/webcrawl2_movie%20details.ipynb)
- diprog's [tls-client-async](https://github.com/diprog/python-tls-client-async) for better URL fetching...

Artist list used: `slushwave-bandcamp-links.txt` compiled in Slushwave Social Club Discord server.

In [1]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import logging
from rich.logging import RichHandler

FORMAT = '%(message)s'
logging.basicConfig(
    level='INFO, format=FORMAT, datefmt='[%X]', handlers=[RichHandler()]
)
log = logging.getLogger('rich')

BASE_DIR = Path.cwd()
LINKS_FILE = BASE_DIR / 'slushwave-bandcamp-links.txt'
OUTPUT_FILE = BASE_DIR / 'bc-albums-image-links.csv'
IMAGES_DIR = BASE_DIR / 'images'

### Use `tls-client` instead of plain requests
TLS fingerprinting gives a more browser-like behavior to bypass anti-bot protections.

In [2]:
from headers import HEADERS_POOL
from bs4 import BeautifulSoup
from async_tls_client import AsyncSession
import asyncio
import os
import random

In [3]:
class BrowserSession:
	def __init__(self):
		self.requests_made = 0
		self.rotate_headers()
		self.retire_after = random.randint(40, 100)

	def rotate_headers(self, headers=None):
		self.session = AsyncSession(
			client_identifier='chrome120',
			random_tls_extension_order=True
		)
		self.session.proxies.update({'http': os.getenv('mobileproxyuk'), 'https': os.getenv('mobileproxyuk')})
		if headers:
			self.session.headers.update(headers)

	async def get(self, url, **kwargs):
		if self.requests_made >= self.retire_after:
			self.rotate_headers()
			self.requests_made = 0
			self.retire_after = random.randint(40, 100)

		resp = await self.session.get(url,**kwargs)
		self.requests_made += 1
		return resp

In [197]:
import re
def nozero(text):
	return re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)

In [ ]:
url1 = 'https://daysofblue.bandcamp.com/album/--12'
url2 = 'https://noproblematapes.bandcamp.com/album/--89'
random.seed(42)
s = BrowserSession()
r1 = await s.get(url1)
soup1 = BeautifulSoup(r1.text, 'lxml')
r2 = await s.get(url2)
soup2 = BeautifulSoup(r2.text, 'lxml')

In [198]:
url3 = 'https://geometriclullaby.bandcamp.com/album/geo-c07'
r3 = await s.get(url3)
soup3 = BeautifulSoup(r3.text, 'lxml')

In [199]:
import json
album_schema = json.loads(soup3.select("script[type='application/ld+json']")[0].get_text(strip=True))
tralbum_tag = soup3.select_one("[data-tralbum]")

[nozero(album_schema['name']), # album name
 nozero(album_schema['byArtist']['name']), # artist name
 album_schema['numTracks'], # number of tracks
 album_schema['keywords'] # tags
]

['GEO - C07; 夕暮れから夜明けまで',
 'Illusionary ドリーミング & ネザーnether',
 36,
 ['Electronic',
  'ambient',
  'dark ambient',
  'dreampunk',
  'dreamtone',
  'experimental',
  'hypnagogic',
  'liminal',
  'liminal spaces',
  'post-vaporwave',
  'slushwave',
  'vaporwave',
  'Pittsburgh']]

In [187]:
# Get all track urls
import isodate
import pandas as pd
df = pd.DataFrame([
	{
		"url": t['item']['@id']
	}
	for t in album_schema["track"]["itemListElement"]
])
print(df)

                                                  url
0   https://geometriclullaby.bandcamp.com/track/--...
1   https://geometriclullaby.bandcamp.com/track/--...
2   https://geometriclullaby.bandcamp.com/track/--...
3   https://geometriclullaby.bandcamp.com/track/--...
4   https://geometriclullaby.bandcamp.com/track/--...
5   https://geometriclullaby.bandcamp.com/track/--...
6   https://geometriclullaby.bandcamp.com/track/--...
7   https://geometriclullaby.bandcamp.com/track/--...
8   https://geometriclullaby.bandcamp.com/track/--...
9   https://geometriclullaby.bandcamp.com/track/--...
10  https://geometriclullaby.bandcamp.com/track/--...
11  https://geometriclullaby.bandcamp.com/track/--...
12  https://geometriclullaby.bandcamp.com/track/--...
13  https://geometriclullaby.bandcamp.com/track/--...
14  https://geometriclullaby.bandcamp.com/track/--...
15  https://geometriclullaby.bandcamp.com/track/--...
16  https://geometriclullaby.bandcamp.com/track/--...
17  https://geometriclullaby

#### Get unique track arts
This is how bandcamp art ID works:
1. Bandcamp generates a unique art_ID for each time you upload a track art.
2. The rest uses art_ID from the album art..

We can drop duplicates based on image hash, with some cases we have to delete manually (dobs album).

In [189]:
import re
import hashlib

async def get_art_id(url):
	session = BrowserSession()
	r = await session.get(url)
	soup = BeautifulSoup(r.text or "", "lxml")

	icon_url = soup.select_one('link[rel="shortcut icon"]')['href']

	img = await session.get(icon_url)
	img_hash = hashlib.sha256(img.content).hexdigest()

	art_id = re.search(r'a(\d+)_', icon_url).group(1)

	return art_id, img_hash


results = await asyncio.gather(
    *(get_art_id(url) for url in df["url"])
)

art_df = pd.DataFrame(
    results,
    columns=["art_id", "img_hash"]
)

print(art_df)

        art_id                                           img_hash
0   3516630582  1aa86359b7c3618b0b5d5cd76250a3a9e3a5631ed497a2...
1   2869282147  993465fe41d27d9311583bcdb30bd89dbe867527622cd3...
2   0964816122  993465fe41d27d9311583bcdb30bd89dbe867527622cd3...
3   2340923502  993465fe41d27d9311583bcdb30bd89dbe867527622cd3...
4   0318286410  993465fe41d27d9311583bcdb30bd89dbe867527622cd3...
5   4025291362  993465fe41d27d9311583bcdb30bd89dbe867527622cd3...
6   0949674429  c66bee984efc2dae02b18a5b7d8fe2dbc7b64f53c94d46...
7   3855028593  c66bee984efc2dae02b18a5b7d8fe2dbc7b64f53c94d46...
8   2498083862  c66bee984efc2dae02b18a5b7d8fe2dbc7b64f53c94d46...
9   0077896647  c66bee984efc2dae02b18a5b7d8fe2dbc7b64f53c94d46...
10  0383452963  c66bee984efc2dae02b18a5b7d8fe2dbc7b64f53c94d46...
11  0290205804  c66bee984efc2dae02b18a5b7d8fe2dbc7b64f53c94d46...
12  0379081337  7eaa223b98fc59764b33d1c989c1a1b75137804a020a44...
13  0712720415  7eaa223b98fc59764b33d1c989c1a1b75137804a020a44...
14  411741

In [190]:
print(f"Total unique art IDs: {art_df['art_id'].nunique()}, Unique Image Hashes: {art_df['img_hash'].nunique()}")

Total unique art IDs: 36, Unique Image Hashes: 7


#### Get all dates of the album

In [191]:
tralbum = json.loads(tralbum_tag["data-tralbum"])['current']
tralbum2 = json.loads(tralbum_tag["data-tralbum"])['trackinfo']

In [200]:
[nozero(tralbum['title']), # album name
 tralbum['art_id'], # album image id: 2569838638 in https://f4.bcbits.com/img/a2569838638_10.jpg
 tralbum['new_date'], # date first created draft
 tralbum['mod_date'], # date last modified
 tralbum['publish_date'], # publication date
 tralbum['release_date'] # release date
 ]

['GEO - C07; 夕暮れから夜明けまで',
 3195284159,
 '28 Nov 2024 03:05:29 GMT',
 '16 Jul 2025 02:17:10 GMT',
 '11 Jul 2025 16:00:08 GMT',
 '11 Jul 2025 00:00:00 GMT']

In [193]:
# Get all track durations
import pandas as pd
track_df = pd.DataFrame([
	{
		"position": t['track_num'],
		"duration": t['duration'],
	}
	for t in tralbum2
])
print(track_df)

    position  duration
0          1   361.230
1          2   579.472
2          3   335.222
3          4   179.172
4          5   743.036
5          6  1379.260
6          7   513.500
7          8   354.449
8          9   600.473
9         10   192.170
10        11   507.199
11        12  1174.920
12        13   366.126
13        14   433.053
14        15   223.036
15        16   665.974
16        17   483.418
17        18  1118.100
18        19   448.626
19        20   541.331
20        21   166.661
21        22   492.807
22        23   566.374
23        24  1118.930
24        25   408.654
25        26   525.779
26        27   363.490
27        28   659.293
28        29   655.072
29        30   355.808
30        31   302.067
31        32   182.615
32        33   472.596
33        34   371.942
34        35   504.591
35        36  1135.250


#### Adding up total runtime of the album:

In [194]:
from datetime import timedelta
print(timedelta(seconds=int(track_df["duration"].sum())))

5:24:41
